# CatBoost: Predicting Transit Time (Track A)

Predicts `transit_days` for each shipment. The goal from the project overview is to beat the naive "predict the promised date" baseline (MAE about 0.84 days) by a meaningful margin, and to hold up across modes and lanes.

- **Data:** `engineered_features.csv`, produced by `clean_data.ipynb`. Run that notebook first.
- **Metrics:** MAE (primary), RMSE, and the share of predictions within ±1 day.
- `test.csv` is held out for the finished model and is not used here. The test slice below is carved out of `train.csv`.

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split

SEED = 1234

df = pd.read_csv("engineered_features.csv", dtype={"origin_zip": str, "dest_zip": str})
assert df["shipment_id"].is_unique
df.shape

(39747, 37)

## 1. Train / validation / test split

Random 70 / 15 / 15 split. Validation is used for every modelling choice. The test slice is scored once, at the very end. Other models should use the same `SEED` and proportions so results are directly comparable.

In [2]:
train, holdout = train_test_split(df, test_size=0.30, random_state=SEED)
val, test = train_test_split(holdout, test_size=0.50, random_state=SEED)

assert len(train) + len(val) + len(test) == len(df)
len(train), len(val), len(test)

(27822, 5962, 5963)

## 2. Baseline

- **Promised date:** predict `promised_days`. This is the bar set in the overview.

In [3]:
TARGET = "transit_days"

def score(y_true, y_pred):
    errors = np.asarray(y_true) - np.asarray(y_pred)
    return {
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt((errors ** 2).mean()),
        "within_1_day_pct": 100 * (np.abs(errors) <= 1).mean(),
    }

pd.DataFrame({
    "promised_date": score(val[TARGET], val["promised_days"]),
}).T.round(4)

,MAE,RMSE,within_1_day_pct
promised_date,0.8366,1.151,84.5689


## 3. Features

IDs, raw dates, ZIPs and raw coordinates are left out: the metro and distance columns already carry that information. `delivered_on_time` is excluded because it's derived from the target, so using it would be leakage.

In [4]:
CATS = ["shipment_mode", "carrier", "service_level", "origin_metro", "dest_metro"]
NUMS = ["ship_dow", "ship_month", "distance_miles", "zone",
        "weight_lbs", "length_in", "width_in", "height_in", "pallet_count",
        "residential_flag", "signature_required", "declared_value_usd",
        "weight_lbs_na", "length_in_na", "width_in_na", "height_in_na",
        "fuel_surcharge_pct", "weather_severity", "carrier_congestion_index",
        "promised_days", "carrier_cost_usd"]
FEATURES = CATS + NUMS

excluded = {TARGET, "delivered_on_time"}
assert not excluded & set(FEATURES), "target leaked into features"
assert set(FEATURES) <= set(df.columns)

## 4. Compare targets

The loss is MAE because that's what the project is scored on. Early stopping on validation picks the number of trees.

Answered on validation: is it better to predict `transit_days` directly, or the delay (`transit_days - promised_days`) and add the promise back?

In [5]:
def fit_catboost(predict_delay=False, **params):
    def target(data):
        return data[TARGET] - data["promised_days"] if predict_delay else data[TARGET]

    settings = dict(iterations=1500, learning_rate=0.08, depth=6, loss_function="MAE", eval_metric="MAE",
                    early_stopping_rounds=100, random_seed=SEED, verbose=0, allow_writing_files=False)
    settings.update(params)
    model = CatBoostRegressor(**settings)
    model.fit(Pool(train[FEATURES], target(train), cat_features=CATS),
              eval_set=Pool(val[FEATURES], target(val), cat_features=CATS))
    return model

def predict_days(model, data, predict_delay=False):
    preds = model.predict(data[FEATURES])
    return preds + data["promised_days"].values if predict_delay else preds

rows = []
for predict_delay in [False, True]:
    model = fit_catboost(predict_delay)
    preds = predict_days(model, val, predict_delay)
    rows.append({"target": "delay" if predict_delay else "transit_days",
                 **score(val[TARGET], preds), "trees": model.tree_count_})

comparison = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
comparison.round(4)

,target,MAE,RMSE,within_1_day_pct,trees
0,transit_days,0.5375,0.7805,87.5042,193
1,delay,0.5380,0.7814,87.3532,372


## 5. Light tuning

A small grid on the best setup from section 4. It's kept small so validation isn't overfit by trying too many options.

In [6]:
best_delay = comparison.loc[0, "target"] == "delay"
print("tuning target:", comparison.loc[0, "target"])

rows = []
for depth in [4, 6, 8]:
    for learning_rate in [0.03, 0.08]:
        model = fit_catboost(best_delay, depth=depth, learning_rate=learning_rate)
        preds = predict_days(model, val, best_delay)
        rows.append({"depth": depth, "learning_rate": learning_rate, **score(val[TARGET], preds), "trees": model.tree_count_})

tuning = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
tuning.round(4)

tuning target: transit_days


,depth,learning_rate,MAE,RMSE,within_1_day_pct,trees
0,6,0.03,0.5366,0.7790,87.5042,838
1,4,0.03,0.5372,0.7771,87.6384,1311
2,6,0.08,0.5375,0.7805,87.5042,193
3,8,0.03,0.5377,0.7815,87.5713,669
4,4,0.08,0.5381,0.7779,87.7390,341
5,8,0.08,0.5394,0.7827,87.4371,345


## 6. Final model and rounding

`transit_days` is always a whole number, so this checks on validation whether rounding the prediction helps.

In [7]:
final_model = fit_catboost(best_delay, depth=int(tuning.loc[0, "depth"]), learning_rate=float(tuning.loc[0, "learning_rate"]))

def predict_final(data, rounded):
    preds = predict_days(final_model, data, best_delay)
    return np.round(preds).clip(min=1) if rounded else preds

pd.DataFrame({
    "unrounded": score(val[TARGET], predict_final(val, rounded=False)),
    "rounded": score(val[TARGET], predict_final(val, rounded=True)),
}).T.round(4)

,MAE,RMSE,within_1_day_pct
unrounded,0.5366,0.7790,87.5042
rounded,0.4775,0.8302,96.3771


## 7. Final score on the test slice

Every choice above was made on validation, so this is the honest estimate of performance. Don't go back and change the model based on this table, or it stops being one.

In [8]:
test_results = pd.DataFrame({
    "promised_date": score(test[TARGET], test["promised_days"]),
    "catboost": score(test[TARGET], predict_final(test, rounded=False)),
    "catboost_rounded": score(test[TARGET], predict_final(test, rounded=True)),
}).T
test_results.round(4)

,MAE,RMSE,within_1_day_pct
promised_date,0.8253,1.1320,85.0075
catboost,0.5314,0.7520,87.8752
catboost_rounded,0.4684,0.8013,96.4783


## 8. Error analysis

Where is the model weakest, and does it beat the promised date everywhere? All tables use unrounded predictions on the test slice.

In [9]:
errors = test.assign(
    lane=test["origin_metro"] + " -> " + test["dest_metro"],  # grouping label for the lane table only, not a model feature
    model_error=np.abs(test[TARGET] - predict_final(test, rounded=False)),
    promised_error=np.abs(test[TARGET] - test["promised_days"]),
)

def error_by(col):
    table = errors.groupby(col).agg(
        shipments=("model_error", "size"),
        model_MAE=("model_error", "mean"),
        promised_MAE=("promised_error", "mean"),
        within_1_day_pct=("model_error", lambda e: 100 * (e <= 1).mean()),
    )
    return table.sort_values("model_MAE", ascending=False).round(3)

for col in ["shipment_mode", "carrier", "service_level", "zone", "ship_month"]:
    display(error_by(col))

,shipments,model_MAE,promised_MAE,within_1_day_pct
shipment_mode,,,,
parcel,5099,0.535,0.833,87.919
ltl_freight,864,0.510,0.777,87.616


,shipments,model_MAE,promised_MAE,within_1_day_pct
carrier,,,,
Falcon Post,2061,0.641,0.859,81.223
Ironclad Freight,505,0.555,0.729,84.158
Northstar Parcel,1672,0.517,0.661,89.713
Vela Freight,359,0.446,0.844,92.479
Cobalt Regional,177,0.435,0.729,93.785
Summit Express,1189,0.392,1.047,96.131


,shipments,model_MAE,promised_MAE,within_1_day_pct
service_level,,,,
TwoDay,770,0.552,0.827,85.584
Ground,2830,0.537,0.844,87.668
Overnight,526,0.529,0.816,88.213
Expedited,973,0.520,0.818,90.339
LTL Standard,647,0.515,0.811,87.635
LTL Guaranteed,217,0.495,0.673,87.558


,shipments,model_MAE,promised_MAE,within_1_day_pct
zone,,,,
7,989,0.558,0.892,86.047
3,239,0.556,0.854,89.121
5,1774,0.537,0.874,88.444
4,656,0.525,0.761,88.415
8,1111,0.519,0.773,87.039
6,1174,0.512,0.779,88.756
2,20,0.486,0.550,90.000


,shipments,model_MAE,promised_MAE,within_1_day_pct
ship_month,,,,
4,507,0.620,0.895,84.615
5,513,0.536,0.819,87.719
6,502,0.536,0.789,87.251
10,553,0.535,0.886,87.703
9,491,0.532,0.796,87.169
2,466,0.530,0.809,87.554
1,506,0.520,0.767,88.933
12,494,0.520,0.846,86.842
3,499,0.514,0.796,90.180


In [10]:
# Lanes need enough test shipments for their MAE to mean anything
MIN_LANE_SHIPMENTS = 15

lanes = error_by("lane").query("shipments >= @MIN_LANE_SHIPMENTS")
print(f"{len(lanes)} lanes with at least {MIN_LANE_SHIPMENTS} test shipments")
print(f"lanes where the model beats the promised date: {(lanes['model_MAE'] < lanes['promised_MAE']).sum()}")
print(f"lane MAE range: {lanes['model_MAE'].min():.3f} to {lanes['model_MAE'].max():.3f}")
lanes.head(10)

204 lanes with at least 15 test shipments
lanes where the model beats the promised date: 202
lane MAE range: 0.175 to 1.019


,shipments,model_MAE,promised_MAE,within_1_day_pct
lane,,,,
Charlotte NC -> Los Angeles CA,17,1.019,1.294,76.471
Chicago IL -> Salt Lake City UT,16,0.959,1.188,68.750
Portland OR -> Chicago IL,19,0.914,1.263,78.947
Atlanta GA -> Boston MA,21,0.844,1.238,90.476
Denver CO -> Boston MA,15,0.839,1.067,86.667
Charlotte NC -> New York NY,17,0.771,0.941,82.353
New York NY -> Nashville TN,18,0.769,0.889,83.333
Boston MA -> New York NY,22,0.760,1.091,86.364
Boston MA -> Dallas TX,22,0.742,1.136,72.727


In [11]:
pd.Series(final_model.get_feature_importance(), index=FEATURES, name="importance") \
    .sort_values(ascending=False).head(15).round(2)

promised_days               47.38
carrier                     16.45
service_level                8.03
zone                         6.00
weather_severity             5.01
carrier_cost_usd             4.33
ship_dow                     4.20
distance_miles               3.93
ship_month                   1.13
carrier_congestion_index     1.12
residential_flag             0.46
fuel_surcharge_pct           0.44
weight_lbs                   0.41
length_in                    0.21
height_in                    0.18
Name: importance, dtype: float64

## 9. Results

Test slice, `SEED = 1234`:

| Model | MAE | RMSE | Within ±1 day |
|---|---|---|---|
| Promised date | 0.825 | 1.132 | 85.0% |
| CatBoost | 0.531 | 0.752 | 87.9% |
| CatBoost, rounded | 0.468 | 0.801 | 96.5% |

- **It clears the bar:** rounded MAE is about 43% below the promised-date baseline. Rounding lowers MAE, but makes RMSE slightly worse. Its jump in the ±1 day share is partly mechanical, because rounding pulls near misses inside the window.
- **Modes:** parcel and freight perform about the same (MAE 0.535 and 0.510).
- **Weakest carrier:** Falcon Post (MAE 0.641, 81.2% within ±1 day) has noticeably more unexplained variation than the others.
- **Lanes:** the model beats the promised date on 202 of the 204 lanes with at least 15 test shipments. Lane-level figures rest on few shipments, so treat the extremes with caution.
- **Predicting delay didn't help:** it was no better than predicting `transit_days` directly.
- **Limitation:** this is a random split. A time-ordered split would be a tougher test of how the model handles future shipments.